# Universal Data Cleaner

A reusable Python tool for cleaning datasets before Machine Learning.

## Project Purpose

The purpose of this project is to improve dataset quality before training machine learning models.

It identifies missing values, duplicate rows, categorical data, and potential outliers.

It also generates reports to explain the quality of the original and cleaned datasets.

## Main Objectives

- Analyze dataset quality.
- Handle missing values using median and mode.
- Detect potential outliers using the IQR method.
- Cap outliers when configured.
- Process categorical features.
- Handle duplicate rows safely.
- Generate data quality reports.
- Prepare datasets for machine learning models.
- Prevent data leakage between training and testing data.

## Future Applications

This cleaner will be reusable with Polynomial Regression, L1 (Lasso), L2 (Ridge), and Elastic Net models.

In [1]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Libraries imported successfully!")

Pandas version: 3.0.6
NumPy version: 2.5.4
Libraries imported successfully!


## Step 2: Create a Sample Dataset

We will create a sample dataset containing numerical and categorical features.

The dataset intentionally includes:

- Missing values
- Duplicate rows
- Potential outliers
- Numerical columns
- Categorical columns

This dataset will help us test whether our Universal Data Cleaner works correctly.

The original dataset will remain unchanged so that we can compare it with the cleaned dataset later.

In [2]:
data = {
    "Age": [22, 25, 28, 35, 40, 29, 31, np.nan, 26, 33, 120, 22],
    "Salary": [30000, 35000, 42000, 55000, 65000, np.nan, 48000, 52000, 39000, 60000, 5000000, 30000],
    "Experience": [1, 2, 4, 10, 15, 5, np.nan, 8, 3, 9, 70, 1],
    "Department": ["IT", "HR", "IT", "Finance", "HR", "IT", None, "Finance", "IT", "HR", "Finance", "IT"]
}

df = pd.DataFrame(data)

print("Original Dataset:")
display(df)

Original Dataset:


,Age,Salary,Experience,Department
0,22.0,30000.0,1.0,IT
1,25.0,35000.0,2.0,HR
2,28.0,42000.0,4.0,IT
3,35.0,55000.0,10.0,Finance
4,40.0,65000.0,15.0,HR
5,29.0,NaN,5.0,IT
6,31.0,48000.0,NaN,NaN
7,NaN,52000.0,8.0,Finance
8,26.0,39000.0,3.0,IT
9,33.0,60000.0,9.0,HR


## Step 3: Data Quality Analysis

Before cleaning a dataset, we need to understand its quality.

In this step, we will check:

- Total number of rows and columns
- Missing values in each column
- Missing value percentages
- Duplicate rows
- Data types
- Unique values in each column

This analysis helps us identify data quality problems before applying any cleaning operations.

In [3]:
total_rows = len(df)
total_columns = len(df.columns)

quality_report = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (df.isnull().mean() * 100).round(2),
    "Unique Values": df.nunique(dropna=True)
})

print("Dataset Shape:", df.shape)
print("Total Rows:", total_rows)
print("Total Columns:", total_columns)
print("Duplicate Rows:", df.duplicated().sum())

print("\nData Quality Report:")
display(quality_report)

Dataset Shape: (12, 4)
Total Rows: 12
Total Columns: 4
Duplicate Rows: 1

Data Quality Report:


,Data Type,Missing Values,Missing Percentage,Unique Values
Age,float64,1,8.33,10
Salary,float64,1,8.33,10
Experience,float64,1,8.33,10
Department,str,1,8.33,3


## Step 4: Outlier Detection Using IQR

Outliers are numerical values that are unusually far from the other values in a dataset.

We will use the Interquartile Range (IQR) method to identify potential outliers.

- Q1 represents the 25th percentile.
- Q3 represents the 75th percentile.
- IQR is calculated as Q3 minus Q1.
- Lower Bound = Q1 - (1.5 × IQR)
- Upper Bound = Q3 + (1.5 × IQR)

Values outside these bounds will be flagged as potential outliers.

Important: An outlier is not always an error. This step only identifies unusual values; it does not modify or remove them."

In [4]:
numeric_columns = df.select_dtypes(include=np.number).columns

outlier_results = []

for column in numeric_columns:
    values = df[column].dropna()

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    outlier_count = (
        (values < lower_bound) | (values > upper_bound)
    ).sum()

    outlier_results.append({
        "Column": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": int(outlier_count)
    })

outlier_report = pd.DataFrame(outlier_results)

print("Potential Outlier Report:")
display(outlier_report.round(2))

Potential Outlier Report:


,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count
0,Age,25.5,34.0,8.5,12.75,46.75,1
1,Salary,37000.0,57500.0,20500.0,6250.00,88250.00,1
2,Experience,2.5,9.5,7.0,-8.00,20.00,1


## Step 5: Build the Reusable DataCleaner Class

We will create a reusable Python class that learns how to clean a dataset.

The cleaner will:

- Learn numerical and categorical columns.
- Calculate numerical medians from training data.
- Learn the most frequent value for categorical columns.
- Calculate IQR boundaries for potential outliers.
- Store the learned cleaning information for later use.

The class will separate learning from cleaning. We will fit it on training data and then use the same learned settings to transform other data.

This helps prevent data leakage in machine learning workflows.

In [5]:
class DataCleaner:

    def __init__(self, cap_outliers=True):
        self.cap_outliers = cap_outliers

    def fit(self, df):
        self.columns = df.columns.tolist()

        self.numeric_columns = df.select_dtypes(
            include=np.number
        ).columns.tolist()

        self.categorical_columns = df.select_dtypes(
            exclude=np.number
        ).columns.tolist()

        self.medians = {}
        self.categories = {}
        self.lower_bounds = {}
        self.upper_bounds = {}

        for column in self.numeric_columns:
            values = df[column].dropna()

            self.medians[column] = (
                values.median() if not values.empty else 0
            )

            if not values.empty:
                q1 = values.quantile(0.25)
                q3 = values.quantile(0.75)
                iqr = q3 - q1

                self.lower_bounds[column] = q1 - 1.5 * iqr
                self.upper_bounds[column] = q3 + 1.5 * iqr
            else:
                self.lower_bounds[column] = -np.inf
                self.upper_bounds[column] = np.inf

        for column in self.categorical_columns:
            values = df[column].dropna()

            self.categories[column] = (
                values.mode().iloc[0] if not values.empty else "Unknown"
            )

        return self

    def transform(self, df):
        cleaned = df.reindex(columns=self.columns).copy()

        for column in self.numeric_columns:
            cleaned[column] = pd.to_numeric(
                cleaned[column], errors="coerce"
            )

            cleaned[column] = cleaned[column].fillna(
                self.medians[column]
            )

            if self.cap_outliers:
                cleaned[column] = cleaned[column].clip(
                    lower=self.lower_bounds[column],
                    upper=self.upper_bounds[column]
                )

        for column in self.categorical_columns:
            cleaned[column] = cleaned[column].fillna(
                self.categories[column]
            )

        return cleaned

    def fit_transform(self, df):
        return self.fit(df).transform(df)


print("DataCleaner class created successfully!")

DataCleaner class created successfully!
